In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)
#/root/.cache/kagglehub/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/versions/1

100%|██████████| 25.7M/25.7M [00:08<00:00, 3.13MB/s]


Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/versions/1


In [2]:
def shorten_column_names(columns):
    """
    Shortens the column names based on predefined mappings.

    Args:
        columns (list of str): List of column names to shorten.

    Returns:
        list of str: List of shortened column names.
    """
    # Define a dictionary of replacements for shortening
    replacements = {
        'Informational Social Support (P=1/A=0)': 'Info',
        'Emotional Social Support (P=1/A=0)': 'Emo',
        'Esteem Social Support (P=1/A=0)': 'Esteem',
        'Network Social Support (P=1/A=0)': 'Net',
        'Tangible Social Support (P=1/A=0)': 'Tang'
    }

    # Replace long names with shortened names
    shortened_columns = [replacements.get(col, col) for col in columns]
    return shortened_columns

# Example usage
columns = [
    'Informational Social Support (P=1/A=0)',
    'Emotional Social Support (P=1/A=0)',
    'Esteem Social Support (P=1/A=0)',
    'Network Social Support (P=1/A=0)',
    'Tangible Social Support (P=1/A=0)'
]

shortened_columns = shorten_column_names(columns)
print(shortened_columns)

['Info', 'Emo', 'Esteem', 'Net', 'Tang']


In [8]:
data=pd.read_excel('f1_reply_db_all1.xlsx')
data.columns=shorten_column_names(data.columns)
data.columns


Index(['Searching means', 'Douban group', 'Group name', 'Title', 'Type',
       'User ID', 'Content', 'Pubtime/+place', 'Info', 'Emo', 'Esteem', 'Net',
       'Tang', 'sum_', 'Social'],
      dtype='object')

In [31]:
def load_imdb_data(df):
    texts = df['Content'].tolist()
    label_columns = ['Info', 'Emo', 'Esteem', 'Net', 'Tang']
    labels = df[label_columns].astype(int).values.tolist()
    for i, t in enumerate(texts):
        if t is None or not isinstance(t, str) or t.strip() == "":
            print(f"异常文本在第 {i} 条:", repr(t))
    texts = [t if isinstance(t, str) and t.strip() != "" and not t.strip().isdigit() else "[NUMERIC]" for t in texts]
    def int_to_emoji(t):
        if isinstance(t, int):
            try:
                return chr(t)
            except ValueError:
                return ""  # 无法转成 emoji 的数字
        return t
    
    data['Content '] = [int_to_emoji(t) if not isinstance(t, str) else t for t in texts]
    return texts, labels

#data_file ='/root/.cache/kagglehub/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/versions/1/IMDB Dataset.csv'
#load_imdb_data(data_file)
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            return_tensors='pt',
            max_length=self.max_length,
            padding='max_length',
            truncation=True
        )
        # 注意：encoding['input_ids'] shape 是 [1, max_length]，flatten后变成 [max_length]
        print(type(text), text)
        return {
            'input_ids': encoding['input_ids'].squeeze(0),  # 或 flatten()
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)  # 多标签时可能是 torch.float
        }
class BERTClassifier(nn.Module):
    def __init__(self, bert_model_name, num_classes):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        x = self.dropout(pooled_output)
        logits = self.fc(x)
        return logits
from tqdm import tqdm
import torch
import torch.nn as nn

def train(model, data_loader, optimizer, scheduler, device):
    model.train()  # 启用训练模式
    total_loss = 0  # 用于累计损失
    for batch in tqdm(data_loader, desc="Training", leave=False):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)  # 注意是 'label'，和你的数据结构保持一致

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = nn.BCEWithLogitsLoss()(outputs, labels.float())
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(data_loader)
    print(f"Average training loss: {avg_loss:.4f}")
    return avg_loss
from sklearn.metrics import accuracy_score, classification_report, f1_score

def evaluate(model, data_loader, device):
    model.eval()
    predictions = []
    actual_labels = []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)  # [batch, num_labels]
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            # 假设 outputs 形状为 [batch, num_labels]
            preds = (outputs.sigmoid() > 0.5).int()  # 得到0/1预测
            predictions.append(preds.cpu())
            actual_labels.append(labels.cpu())

    # 拼接所有batch
    predictions = torch.cat(predictions, dim=0).numpy()
    actual_labels = torch.cat(actual_labels, dim=0).numpy()

    # 多标签分类常用f1_score（macro/micro/weighted都可以）
    f1 = f1_score(actual_labels, predictions, average="micro")
    report = classification_report(actual_labels, predictions, zero_division=0)
    return f1, report

def predict_sentiment(text, model, tokenizer, device, max_length=128):
    model.eval()
    encoding = tokenizer(text, return_tensors='pt', max_length=max_length, padding='max_length', truncation=True)
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs, dim=1)
    return preds.item()

In [19]:
# Set up parameters
bert_model_name = 'bert-base-uncased'
num_classes = 5
max_length = 128
batch_size = 16
num_epochs = 4
learning_rate = 2e-5


In [12]:
#Loading and splitting the data.
texts,labels=load_imdb_data(data)
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)


异常文本在第 385 条: 1127596715
异常文本在第 387 条: 18213086586
异常文本在第 409 条: 15070213637
异常文本在第 515 条: 13968899693


In [13]:
bert_model_name = "autodl-tmp/bert_chinese"  # 这是模型的名称字符串
tokenizer = BertTokenizer.from_pretrained(bert_model_name,cache_dir="autodl-tmp",  # 可选，指定本地缓存目录
    mirror='https://hf-mirror.com')


In [14]:
train_dataset = TextClassificationDataset(train_texts, train_labels, tokenizer, max_length)
val_dataset = TextClassificationDataset(val_texts, val_labels, tokenizer, max_length)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BERTClassifier("autodl-tmp/bert_chinese", num_classes).to(device)

In [16]:
optimizer = AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_dataloader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

/root/miniconda3/lib/python3.10/site-packages/transformers/optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [17]:

print(type(labels), labels.shape)
print(type(preds), preds.shape)


AttributeError: 'list' object has no attribute 'shape'

In [20]:
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    train(model, train_dataloader, optimizer, scheduler, device)
    accuracy, report = evaluate(model, val_dataloader, device)
    print(f"Validation Accuracy: {accuracy:.4f}")
    print(report)

Epoch 1/4


Training:   7%|▋         | 2/30 [00:00<00:02, 13.35it/s]

<class 'str'> 宝这个真的是心态问题 一定要调整过来
<class 'str'> 首先是戒糖（精制糖、添加糖）_x000D__x000D_
戒零食（加工食品）_x000D__x000D_
戒水果（水果也是我的trigger food，而且水果的营养例如VC也是可以通过吃有些生疏菜获得的，比如萝卜，西红柿??）
<class 'str'> 我双相，吃了氟西汀反而开始暴食了……就是你不要抱太大希望吧，而且氟西汀两周才起效。对于抑郁我也觉得氟西汀药劲儿很小。
<class 'str'> 私信你啦
<class 'str'> 要不吃点蛋白粉，维生素啥的
<class 'str'> 别把他的话往心里去，抱抱
<class 'str'> 我也是 连续吐了几天 现在不管怎么刺激都吐不出来 说事身体机能保护自己
<class 'str'> 朋友我也……减肥越减越肥
<class 'str'> 蹲个群
<class 'str'> 我比较喜欢吃东西，你不喜欢我们或许可以互相帮助
<class 'str'> 做好打底，实在吐不出来就算了
<class 'str'> 加我一个ymy110611
<class 'str'> 我前三个月也是这样，再加上上海封城，100天，暴食??催吐??暴食??催吐，整个人在崩溃的边缘，涨了快40斤
<class 'str'> 别吃了，容易脂肪肝??
<class 'str'> 你现在怎么样了，真的这样就戒掉了吗
<class 'str'> 我懂你，各种碳水，我也是。
<class 'str'> 求拉 Charlotte510zz
<class 'str'> 我和你一起吧
<class 'str'> [NUMERIC]
<class 'str'> 我也看完了 楼主说的对 就是因为心理出现了一些问题才把重心寄托到暴食上 有一句话叫做 eat your feelings 我也好想和楼主交流一下怎么才能变好 总是对自己明天以及以后不会再暴食了有着盲目自信 但总是会打破 而且时间长了自己也麻木了 羞耻感和罪恶感都变低 变成了生活中时常出现的一部分 这点很可怕
<class 'str'> 私信留威
<class 'str'> 已自愈，自创断暴疗法，希望帮助更多的姐妹，motanjiemei
<class 'str'> 好的，私你我微信啦，备注一下哈
<cla

Training:  23%|██▎       | 7/30 [00:00<00:01, 18.54it/s]

 ??我和你一样控制饮食到了发疯的地步，可能之前健身的时候有这个控制油的习惯，现在做菜都会有强迫症，怕吃多了油变胖。控制一段时间之后疯狂想吃碳水，一下把想吃的都吃了，吃了又有罪恶感于是就催吐掉，第二天体重掉了一斤，就这样无意识中产生了一个念头，暴食后催吐掉就不会胖了，经过很多次以后可能已经成了习惯无法自拔。
<class 'str'> 亲亲，怎么添加呢
<class 'str'> 摸摸，我暴食完还得忍着难受陪我爸妈吃饭，因为我今年过生日，最后真的绷不住了吃了一点就受不了
<class 'str'> 可以加我一个嘛姐妹，一起
<class 'str'> 小程序:食几圈，戒暴食公益社区和互助交流社群
<class 'str'> vx群的话拉我一下，谢谢
<class 'str'> 我在努力学习。还有写日记也是个好半天，我们要努力呀
<class 'str'> 抱抱 希望我们以后都可以用正常的心境去面对食物 和自己的身体和解 我知道很难 特别难 但也想充满希望和期待
<class 'str'> 抱抱，要么还是和朋友一起在外面住吧？
<class 'str'> 害我这两天又好了，主要是天气太热导致的，这几天连续下雨感觉自己食欲又恢复了hh，我可能这辈子都不会厌食的，想吃的太多了嘻嘻，姐妹也要慢慢好起来啊(??ì _ í??)
<class 'str'> 可以加一下好友吗？想找到群体，不然就一起建个群
<class 'str'> 柚子不胖还会瘦！
<class 'str'> 也许可以试着去看看你的内心，到底为什么有情绪，暴食只是表现和表象，往往是有内在的原因的
<class 'str'> 不算多，但是吃的不太健康，感觉都是零食
<class 'str'> 抱抱你
<class 'str'> 已+
<class 'str'> 能加一下我吗？Rainie970213
<class 'str'> 加油加油！
<class 'str'> 我以前也有最近戒了，就是心态要放下，减肥随缘。
<class 'str'> 我也是！！！！我今天中午又干了四个大馒头十几个红糖肚脐饼
<class 'str'> 我我我我
<class 'str'> 同感，太真实了，只喝得下去酸奶
<class 'str'> 我一般是去宿舍楼的公共厕所吐 单间的那种 隔音效果比较好 或者在吃完饭直接在附近找厕所吐了 

...

Training:  80%|████████  | 24/30 [00:01<00:00, 20.15it/s]

<class 'str'> 真棒姐妹 ！我还在继续和暴食抗争哈哈哈
<class 'str'> 如果今后在一些情况下（例如社交、和别人一起过节日等）吃了这些被禁止的食物，那一定要来这里记录一下
<class 'str'> 摸摸头
<class 'str'> 看了你的帖子哭了，能理解一些又不能理解。食物本该是安慰，是帮助，是美好的。为什么会变成这个样子?? 清清楚楚的看到你知道自己什么该做什么不该做，却又不受控制的去做。希望你还有一丝力气的话，都努力去挣扎，拜托它，战胜它！为你加油！祝福你亲爱的陌生人希望你早点好起来  希望你以前受的伤都能被一点点抚平。
<class 'str'> ??我和你一样控制饮食到了发疯的地步，可能之前健身的时候有这个控制油的习惯，现在做菜都会有强迫症，怕吃多了油变胖。控制一段时间之后疯狂想吃碳水，一下把想吃的都吃了，吃了又有罪恶感于是就催吐掉，第二天体重掉了一斤，就这样无意识中产生了一个念头，暴食后催吐掉就不会胖了，经过很多次以后可能已经成了习惯无法自拔。
<class 'str'> 求拉 Charlotte510zz
<class 'str'> 宝贝，这只是暴饮暴食，是吃多了而已，不要担心
<class 'str'> 我跟你好像 除非有外人在 我自己住就整个放纵
<class 'str'> 亲私信留一下V
<class 'str'> 我也！带带我！
<class 'str'> 你好，我是10年到14年暴食催吐，中度。大概是每一天暴食一次（大学生活费不多买20来块钱的垃圾食品）。后来我痊愈了。讽刺的是我快32岁了，虽然暴食好了，但是腾出来的空间和精力又开始焦虑其他的事情了，生活并没有明朗起来。当然，现在的日子比不暴食好多了。我感觉你的帖子特别真诚，如果你需要我的帮助你可以私信我。_x000D__x000D_
所有的疾病都有路径，找到路径然后退回来一定会痊愈，这不是盲目乐观，是知识的力量。
<class 'str'> 好的，私你我微信啦，备注一下哈
<class 'str'> 加油！！！！一定会好起来的
<class 'str'> 想一起戒吐的留下微信或者QQ一起相互帮助吧
<class 'str'> 用手压嗓子眼就可以，我没看懂sd是啥意思??反正就挺简单的，我个人觉得已经催吐就不要有心理负担了，但是没开始催吐就不必模仿了
<class '

Training:  90%|█████████ | 27/30 [00:01<00:00, 20.15it/s]

<class 'str'> woyes
<class 'str'> 厌食症和催吐很可怕的，上面两个链接详细科普了危害
<class 'str'> 别把他的话往心里去，抱抱
<class 'str'> 爱你亲爱的陌生人
<class 'str'> 已自愈，自创断暴疗法，希望帮助更多的姐妹，motanjiemei
<class 'str'> 我也是
<class 'str'> 抱抱你，调整调整吧，伤身体
<class 'str'> 我也
<class 'str'> 暴食已自愈，希望帮助更多的姐妹motanjiemei
<class 'str'> 姐妹私我微信~我加你
<class 'str'> 唉我每天都有在吃维生素害怕在这种时候生病
<class 'str'> 同感，太真实了，只喝得下去酸奶
<class 'str'> 姐妹求拉，同大三
<class 'str'> 应该正视自己心出了问题，情志出了问题，找路修心，通过传承改本，然后自然而然心不难受了，身体气血和代谢恢复到正常，你的需求和摄入就是正常，想吃什么吃什么，什么时候吃都可以，开开心心，才能真的解决暴食症的问题。
<class 'str'> 姐妹好，我当时是“薄荷网”最开始流行的时候，我入了坑，我是自己买了一个食物称，每天称重，过了没多久就觉得自己是变态，大学4年，现在走出来了。欢迎你来小组。
<class 'str'> 已+
<class 'str'> 首先是戒糖（精制糖、添加糖）_x000D__x000D_
戒零食（加工食品）_x000D__x000D_
戒水果（水果也是我的trigger food，而且水果的营养例如VC也是可以通过吃有些生疏菜获得的，比如萝卜，西红柿??）
<class 'str'> 我愿意当一个 成功案例去解救更多姐妹 你看到的话可以拉我一把
<class 'str'> momocats102
<class 'str'> 举手??
<class 'str'> 楼主可以私信我一下吗
<class 'str'> 没事，放宽心，能涨就能减，长得快，减得也快
<class 'str'> 举手 正在抗争
<class 'str'> 要说容易吐的话就是一边吃一边喝水了，如果吃干的就吐不出来
<class 'str'> 因为节食吧  或者说过分的限制饮食  脑子里面“只能吃健康食品”_x000D__x

Average training loss: 0.3888
<class 'str'> 我也在上海 可以约饭中和一下呀
<class 'str'> 【1.0戒零食??甜食??油炸】以后用这个tag打卡吧，先从1.0开始，比较简单的，应该执行起来不会特别难
<class 'str'> 现在的我…
<class 'str'> 太好了，放心些了_x000D__x000D_
药物主要是用来缓解情绪不适的，_x000D__x000D_
如果医生认为合适，可以看看医院有没有推荐的心理治疗或咨询，配合起来效果是最好的。_x000D__x000D_
吃东西方面可以咨询营养科，一般大医院会有；_x000D__x000D_
更理想的是精神科与营养科联合起来，_x000D__x000D_
推荐北大六院相关科室的科普微信号，有很多有用的信息
<class 'str'> 啊，还可以加么！25
<class 'str'> 抱抱(／≧ω＼)，没关系的，去体会自己真正的情绪与需求，不再用食物解决一切。
<class 'str'> 留微信号加群
<class 'str'> 大家留下微信，我有空加你
<class 'str'> dd
<class 'str'> 我也！我一说我吃的多我有病，我妈就让我别瞎想??
<class 'str'> 同感
<class 'str'> 抱抱 我也暴食好久了 加油 会好起来的
<class 'str'> 举手，间歇性暴食，即使知道自己吃饱了，看到有东西剩下来也是不会停的??
<class 'str'> 嗯 坚持一天就感觉又战胜了自己
<class 'str'> 想結束這無盡的病態食慾 希望可以互助...
<class 'str'> 也不能说光看热量，但是人一天摄入两千卡以内算正常的。 看时间上你是一天分三餐这样吃法，真的就是正常吃正常消耗，如果是全部集中在一餐吃完，那就很暴食了。分人吧也看胃口大小咯
<class 'str'> 你吃的费列罗和特仑苏也足够糖份了
<class 'str'> DD
<class 'str'> 这应该是绝大部分者的心情了，有时候爆完还会莫名其妙的哭，觉得自己好没用
<class 'str'> 我本人??
<class 'str'> 你没有从根源上解决问题
<class 'str'> 留言V
<class 'str'> 嗯嗯嗯！咱们一定可以，

In [32]:
df2=pd.read_excel('remaining_douban.xlsx')
content=[]
for k in range(len(df2)):
    content.append(str(df2['Content'].iloc[k]))
list_lable=[]
for i in content:
    print(i)
    list_lable.append(predict_sentiment(i, model, tokenizer, device))

有 很常
今天意识到经常性的狂躁是因为低血糖
想加入！
我今天一天没吃饭一点都不饿，也不想吃饭，晚上吃了点小橘子，刚刚回来买了条奥利奥喝了一杯燕麦牛奶
我也想进
求带着治病
想进
求拉群
私信你了，，没想到只能发一条信息??
非常需要??
有没有互助群
我是因为心情持续性的暴食，没有任何补偿，5天胖了16斤，现在已经没敢上称
我不吐，吐不出来，但是会补偿运动和第二天少吃。
我
我
开学了，再也没有暴食过了。慢慢地变瘦了一点点。运动量上去了，肌肉也变多了。加入了排球校队，训练特别快乐
我私信你
这
这里小姐姐
我！
我
这里
有
我！
带我一个谢谢
加我一个！_x000D__x000D_
yrn1160875962
加我
楼主私我下吧，想戒掉暴食症，现在已经胖了18斤，中度抑郁症??中度焦虑症了。头发快掉光了
不知道在这里说这个合不合适，但是真的感觉lz开头两段写的很好，有小说的味道哎(????ω????｀)??????
你可以教教我 怎么样才可以不再那么在意体重了吗 要怎么才可以和身体和解 我真的做不到 好难
有意向的也可以加微信cookkies_hei1219
早 三个海绵蛋糕和一瓶牛奶_x000D__x000D_
中 泡面和一份薯条和一份鸡块和一整只鸡和两个葱油饼和两瓶可乐_x000D__x000D_
晚 四个泡芙和一杯柠檬水_x000D__x000D_
今天吃到撑但是没有吃到吐吃到走不动路 刚才去面包店买了三明治作明天的早饭 希望今晚要忍住不要吃掉 希望明天不要吃撑
坐标广州。一年前开始暴饮暴食。一直找不到出路…求伙伴。
前两天控制住了，今天有没控制住，吃了一整盒巧克力黄油曲奇
姐妹，我想加你微信
加拉
男生可以吗？   yangkai_65535
申请加入！最近真的很需要有人一起努力，但还是不敢向周围亲近的人坦白
姐妹，你看看，真的很有用，我原来也是暴食，感觉就是晚上特别控制不住，我就先定个小计划，尽量12点前睡，早点睡就没事，爬起来吃东西，就只吃健康的食品，少买成品，不能一蹴而就，就买点半成品先，而且学会积极暂停????，闭上眼睛，问自己，心????为什么想吃东西
嗯嗯 希望自己好好对待自己的身体 暴食真的太伤害身体了
可以
我也可以
脑子：不能再吃了 嘴巴：一直吃的我来了
你好。。。持续十多年了
你好 我可以
可以嘛我也想
蹲蹲，正在戒
knight_c

In [ ]:
list_lable
#save model
torch.save(model.state_dict(), "bert_classifier.pth")

In [36]:
list_lable
import numpy as np

label_list = list_lable

# 创建全0的数组
arr = np.zeros((len(label_list), 5), dtype=int)

# 将对应列置为1
arr[np.arange(len(label_list)), label_list] = 1

# 转为DataFrame
df = pd.DataFrame(arr, columns=[f'col_{i+1}' for i in range(5)])

print(df)
df.to_excel('remain_lables.xlsx')

      col_1  col_2  col_3  col_4  col_5
0         0      0      0      1      0
1         1      0      0      0      0
2         0      0      0      1      0
3         1      0      0      0      0
4         0      0      0      1      0
...     ...    ...    ...    ...    ...
1178      1      0      0      0      0
1179      1      0      0      0      0
1180      1      0      0      0      0
1181      1      0      0      0      0
1182      1      0      0      0      0

[1183 rows x 5 columns]
